In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Structure Refinement: Si, SEPD

This example demonstrates a Rietveld refinement of Si crystal
structure using time-of-flight neutron powder diffraction data from
SEPD at Argonne.

It also shows how to switch calculation engine and peak profile type.

## Import Library

In [2]:
from easydiffraction import ExperimentFactory
from easydiffraction import Project
from easydiffraction import StructureFactory
from easydiffraction import download_data

## Define Structure

This section shows how to add structures and modify their
parameters.

#### Create Structure

In [3]:
structure = StructureFactory.from_scratch(name='si')

#### Set Space Group

In [4]:
structure.space_group.name_h_m = 'F d -3 m'
structure.space_group.it_coordinate_system_code = '2'

#### Set Unit Cell

In [5]:
structure.cell.length_a = 5.431

#### Set Atom Sites

In [6]:
structure.atom_sites.create(
    label='Si',
    type_symbol='Si',
    fract_x=0.125,
    fract_y=0.125,
    fract_z=0.125,
    adp_iso=0.5,
)

## Define Experiment

This section shows how to add experiments, configure their
parameters, and link the structures defined in the previous step.

#### Download Measured Data

In [7]:
data_path = download_data(id=7, destination='data')

Getting data...


Data #7: Si, SEPD (Argonne)


✅ Data #7 already present at 'data/ed-7.xye'. Keeping existing file.


#### Create Experiment

In [8]:
expt = ExperimentFactory.from_data_path(
    name='sepd', data_path=data_path, beam_mode='time-of-flight'
)

#### Set Instrument

In [9]:
expt.instrument.setup_twotheta_bank = 144.845
expt.instrument.calib_d_to_tof_offset = 0.0
expt.instrument.calib_d_to_tof_linear = 7476.91
expt.instrument.calib_d_to_tof_quad = -1.54

#### Set Peak Profile

In [10]:
expt.show_peak_profile_types()
expt.peak.broad_gauss_sigma_0 = 3.0
expt.peak.broad_gauss_sigma_1 = 40.0
expt.peak.broad_gauss_sigma_2 = 2.0
expt.peak.exp_decay_beta_0 = 0.04221
expt.peak.exp_decay_beta_1 = 0.00946
expt.peak.exp_rise_alpha_0 = 0.0
expt.peak.exp_rise_alpha_1 = 0.5971

Peak profile types


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,,Type,Description
1,,pseudo-voigt,TOF non-convoluted pseudo-Voigt profile
2,*,jorgensen,TOF Jorgensen profile: back-to-back exponentials ⊗ Gaussian
3,,jorgensen-von-dreele,TOF Jorgensen-Von Dreele profile: back-to-back exponentials ⊗ pseudo-Voigt
4,,double-jorgensen-von-dreele,TOF Double-Jorgensen-Von Dreele profile: double back-to-back exponentials ⊗ pseudo-Voigt (Z-Rietveld type0m)


#### Set Background

In [11]:
expt.background_type = 'line-segment'
for x in range(0, 35000, 5000):
    expt.background.create(id=str(x), x=x, y=200)

Background type for experiment 'sepd' already set to


line-segment


#### Set Linked Phases

In [12]:
expt.linked_phases.create(id='si', scale=10.0)

## Define Project

The project object is used to manage the structure, experiment, and
analysis.

#### Create Project

In [13]:
project = Project()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#### Add Structure

In [14]:
project.structures.add(structure)

#### Add Experiment

In [15]:
project.experiments.add(expt)

## Perform Analysis

This section shows the analysis process, including how to set up
calculation and fitting engines.

#### Plot Measured vs Calculated

In [16]:
project.display.plotter.plot_meas_vs_calc(expt_name='sepd', show_residual=True)
project.display.plotter.plot_meas_vs_calc(
    expt_name='sepd', x_min=23200, x_max=23700, show_residual=True
)

### Perform Fit 1/5

Set parameters to be refined.

In [17]:
structure.cell.length_a.free = True

expt.linked_phases['si'].scale.free = True
expt.instrument.calib_d_to_tof_offset.free = True

Show free parameters after selection.

In [18]:
project.analysis.display.free_params()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,si,cell,,length_a,5.43100,,-inf,inf,Å
2,sepd,linked_phases,si,scale,10.00000,,-inf,inf,
3,sepd,instrument,,d_to_tof_offset,0.00000,,-inf,inf,μs


#### Run Fitting

In [19]:
project.analysis.fit()
project.analysis.display.fit_results()

Standard fitting


📋 Using experiment 🔬 'sepd' for 'single' fitting


🚀 Starting fit process with 'lmfit (leastsq)'...


📈 Goodness-of-fit (reduced χ²) change:


,iteration,χ²,improvement [%]
1,1,113.06,
2,7,72.20,36.1% ↓
3,11,66.76,7.5% ↓
4,30,66.72,


...

📏 Weighted R-factor (wR): 12.51%


📈 Fitted parameters:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,datablock,category,entry,parameter,start,fitted,uncertainty,units,change
1,si,cell,,length_a,5.4310,5.4314,0.0002,Å,0.01 % ↑
2,sepd,linked_phases,si,scale,10.0000,13.3619,0.1153,,33.62 % ↑
3,sepd,instrument,,d_to_tof_offset,0.0000,-9.2543,0.2503,μs,N/A


#### Plot Measured vs Calculated

In [20]:
project.display.plotter.plot_meas_vs_calc(expt_name='sepd', show_residual=True)

In [21]:
project.display.plotter.plot_meas_vs_calc(
    expt_name='sepd', x_min=23200, x_max=23700, show_residual=True
)

### Perform Fit 2/5

Set more parameters to be refined.

In [22]:
for point in expt.background:
    point.y.free = True

Show free parameters after selection.

In [23]:
project.analysis.display.free_params()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,si,cell,,length_a,5.43135,0.00018,-inf,inf,Å
2,sepd,linked_phases,si,scale,13.36187,0.11531,-inf,inf,
3,sepd,instrument,,d_to_tof_offset,-9.25432,0.25033,-inf,inf,μs
4,sepd,background,0,y,200.00000,,-inf,inf,
5,sepd,background,5000,y,200.00000,,-inf,inf,
6,sepd,background,10000,y,200.00000,,-inf,inf,
7,sepd,background,15000,y,200.00000,,-inf,inf,
8,sepd,background,20000,y,200.00000,,-inf,inf,
9,sepd,background,25000,y,200.00000,,-inf,inf,
10,sepd,background,30000,y,200.00000,,-inf,inf,


#### Run Fitting

In [24]:
project.analysis.fit()
project.analysis.display.fit_results()

Standard fitting


📋 Using experiment 🔬 'sepd' for 'single' fitting


🚀 Starting fit process with 'lmfit (leastsq)'...


📈 Goodness-of-fit (reduced χ²) change:


,iteration,χ²,improvement [%]
1,1,66.80,
2,14,3.38,94.9% ↓
3,48,3.38,


...

📏 Weighted R-factor (wR): 5.95%


📈 Fitted parameters:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,datablock,category,entry,parameter,start,fitted,uncertainty,units,change
1,si,cell,,length_a,5.4314,5.4314,0.0000,Å,0.00 % ↑
2,sepd,linked_phases,si,scale,13.3619,14.6317,0.0265,,9.50 % ↑
3,sepd,instrument,,d_to_tof_offset,-9.2543,-9.2534,0.0515,μs,0.01 % ↓
4,sepd,background,0,y,200.0000,268.6002,0.9745,,34.30 % ↑
5,sepd,background,5000,y,200.0000,144.7589,0.4071,,27.62 % ↓
6,sepd,background,10000,y,200.0000,120.0247,0.4282,,39.99 % ↓
7,sepd,background,15000,y,200.0000,135.8494,0.8169,,32.08 % ↓
8,sepd,background,20000,y,200.0000,132.6887,1.4317,,33.66 % ↓
9,sepd,background,25000,y,200.0000,175.1775,2.8755,,12.41 % ↓
10,sepd,background,30000,y,200.0000,180.4556,5.8525,,9.77 % ↓


#### Plot Measured vs Calculated

In [25]:
project.display.plotter.plot_meas_vs_calc(expt_name='sepd', show_residual=True)

In [26]:
project.display.plotter.plot_meas_vs_calc(
    expt_name='sepd', x_min=23200, x_max=23700, show_residual=True
)

### Perform Fit 3/5

Fix background points.

In [27]:
for point in expt.background:
    point.y.free = False

Set more parameters to be refined.

In [28]:
expt.peak.broad_gauss_sigma_0.free = True
expt.peak.broad_gauss_sigma_1.free = True
expt.peak.broad_gauss_sigma_2.free = True

Show free parameters after selection.

In [29]:
project.analysis.display.free_params()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,si,cell,,length_a,5.43137,0.00004,-inf,inf,Å
2,sepd,linked_phases,si,scale,14.63167,0.02651,-inf,inf,
3,sepd,peak,,gauss_sigma_0,3.00000,,-inf,inf,μs²
4,sepd,peak,,gauss_sigma_1,40.00000,,-inf,inf,μs/Å
5,sepd,peak,,gauss_sigma_2,2.00000,,-inf,inf,μs²/Å²
6,sepd,instrument,,d_to_tof_offset,-9.25336,0.05153,-inf,inf,μs


#### Run Fitting

In [30]:
project.analysis.fit()
project.analysis.display.fit_results()

Standard fitting


📋 Using experiment 🔬 'sepd' for 'single' fitting


🚀 Starting fit process with 'lmfit (leastsq)'...


📈 Goodness-of-fit (reduced χ²) change:


,iteration,χ²,improvement [%]
1,1,3.38,
2,10,3.21,5.0% ↓
3,39,3.21,


...

📏 Weighted R-factor (wR): 4.88%


📈 Fitted parameters:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,datablock,category,entry,parameter,start,fitted,uncertainty,units,change
1,si,cell,,length_a,5.4314,5.4314,0.0000,Å,0.00 % ↑
2,sepd,linked_phases,si,scale,14.6317,14.7057,0.0257,,0.51 % ↑
3,sepd,peak,,gauss_sigma_0,3.0000,5.7727,0.4206,μs²,92.42 % ↑
4,sepd,peak,,gauss_sigma_1,40.0000,44.2827,0.7966,μs/Å,10.71 % ↑
5,sepd,peak,,gauss_sigma_2,2.0000,1.2962,0.1680,μs²/Å²,35.19 % ↓
6,sepd,instrument,,d_to_tof_offset,-9.2534,-9.2506,0.0546,μs,0.03 % ↓


#### Plot Measured vs Calculated

In [31]:
project.display.plotter.plot_meas_vs_calc(expt_name='sepd', show_residual=True)

In [32]:
project.display.plotter.plot_meas_vs_calc(
    expt_name='sepd', x_min=23200, x_max=23700, show_residual=True
)

### Perform Fit 4/5

Set more parameters to be refined.

In [33]:
structure.atom_sites['Si'].adp_iso.free = True

expt.peak.exp_decay_beta_0.free = True
expt.peak.exp_decay_beta_1.free = True
expt.peak.exp_rise_alpha_1.free = True

Show free parameters after selection.

In [34]:
project.analysis.display.free_params()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,si,cell,,length_a,5.43143,0.00004,-inf,inf,Å
2,si,atom_site,Si,adp_iso,0.50000,,-inf,inf,Å²
3,sepd,linked_phases,si,scale,14.70568,0.02567,-inf,inf,
4,sepd,peak,,rise_alpha_1,0.59710,,-inf,inf,μs/Å
5,sepd,peak,,decay_beta_0,0.04221,,-inf,inf,μs
6,sepd,peak,,decay_beta_1,0.00946,,-inf,inf,μs/Å
7,sepd,peak,,gauss_sigma_0,5.77274,0.42055,-inf,inf,μs²
8,sepd,peak,,gauss_sigma_1,44.28265,0.79664,-inf,inf,μs/Å
9,sepd,peak,,gauss_sigma_2,1.29621,0.16795,-inf,inf,μs²/Å²
10,sepd,instrument,,d_to_tof_offset,-9.25065,0.05458,-inf,inf,μs


#### Run Fitting

In [35]:
project.analysis.fit()
project.analysis.display.fit_results()

Standard fitting


📋 Using experiment 🔬 'sepd' for 'single' fitting


🚀 Starting fit process with 'lmfit (leastsq)'...


📈 Goodness-of-fit (reduced χ²) change:


,iteration,χ²,improvement [%]
1,1,3.21,
2,15,3.17,1.2% ↓
3,16,3.13,1.5% ↓
4,26,3.01,3.7% ↓
5,38,2.96,1.7% ↓
6,60,2.93,1.1% ↓
7,105,2.93,


...

📏 Weighted R-factor (wR): 2.48%


📈 Fitted parameters:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,datablock,category,entry,parameter,start,fitted,uncertainty,units,change
1,si,cell,,length_a,5.4314,5.4325,0.0001,Å,0.02 % ↑
2,si,atom_site,Si,adp_iso,0.5000,0.5240,0.0033,Å²,4.80 % ↑
3,sepd,linked_phases,si,scale,14.7057,14.9693,0.0355,,1.79 % ↑
4,sepd,peak,,rise_alpha_1,0.5971,0.2370,0.0043,μs/Å,60.31 % ↓
5,sepd,peak,,decay_beta_0,0.0422,0.0386,0.0002,μs,8.65 % ↓
6,sepd,peak,,decay_beta_1,0.0095,0.0106,0.0002,μs/Å,11.67 % ↑
7,sepd,peak,,gauss_sigma_0,5.7727,6.9657,0.4577,μs²,20.67 % ↑
8,sepd,peak,,gauss_sigma_1,44.2827,25.6509,1.0250,μs/Å,42.07 % ↓
9,sepd,peak,,gauss_sigma_2,1.2962,1.1001,0.1584,μs²/Å²,15.13 % ↓
10,sepd,instrument,,d_to_tof_offset,-9.2506,-8.7248,0.0740,μs,5.68 % ↓


#### Show parameter correlations

In [36]:
project.display.plotter.plot_param_correlations()

⚠️ No parameter pairs with |correlation| >= 0.70 were found.                                                                      


#### Plot Measured vs Calculated

In [37]:
project.display.plotter.plot_meas_vs_calc(expt_name='sepd', show_residual=True)

In [38]:
project.display.plotter.plot_meas_vs_calc(
    expt_name='sepd', x_min=23200, x_max=23700, show_residual=True
)

In [39]:
project.display.plotter.plot_meas_vs_calc(expt_name='sepd', x='d_spacing', show_residual=True)

### Perform Fit 5/5

#### Switch calculator engine

In [40]:
expt.calculation.show_calculator_types()

Calculator types


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,,Type,Description
1,,crysfml,CrysFML library for crystallographic calculations
2,*,cryspy,CrysPy library for crystallographic calculations


In [41]:
expt.calculation.calculator_type = 'crysfml'

Calculator for experiment 'sepd' changed to


crysfml


#### Change peak profile type

In [42]:
expt.show_peak_profile_types()

Peak profile types


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,,Type,Description
1,*,jorgensen,TOF Jorgensen profile: back-to-back exponentials ⊗ Gaussian
2,,jorgensen-von-dreele,TOF Jorgensen-Von Dreele profile: back-to-back exponentials ⊗ pseudo-Voigt


In [43]:
expt.peak_profile_type = 'jorgensen-von-dreele'

⚠️ Switching peak profile type discards existing peak parameters.                                                                 


Peak profile type for experiment 'sepd' changed to


jorgensen-von-dreele


In [44]:
expt.peak.broad_gauss_sigma_0 = 3.0148
expt.peak.broad_gauss_sigma_1 = 33.3451
expt.peak.broad_lorentz_gamma_1 = 2.5489
expt.peak.exp_decay_beta_0 = 0.04221
expt.peak.exp_decay_beta_1 = 0.00946
expt.peak.exp_rise_alpha_1 = 0.5971

#### Add new free parameters

In [45]:
expt.peak.broad_gauss_sigma_0.free = True
expt.peak.broad_gauss_sigma_1.free = True
expt.peak.broad_lorentz_gamma_1.free = True
expt.peak.exp_decay_beta_0.free = True
expt.peak.exp_decay_beta_1.free = True
expt.peak.exp_rise_alpha_1.free = True

#### Run Fitting

In [46]:
project.analysis.fit()
project.analysis.display.fit_results()

Standard fitting


📋 Using experiment 🔬 'sepd' for 'single' fitting


🚀 Starting fit process with 'lmfit (leastsq)'...


📈 Goodness-of-fit (reduced χ²) change:


,iteration,χ²,improvement [%]
1,1,190.52,
2,40,187.83,1.4% ↓
3,51,185.11,1.4% ↓
4,62,180.34,2.6% ↓
5,73,172.48,4.4% ↓
6,84,160.60,6.9% ↓
7,95,141.26,12.0% ↓
8,106,109.24,22.7% ↓
9,117,66.63,39.0% ↓
10,128,32.60,51.1% ↓


...

📏 Weighted R-factor (wR): 2.41%


📈 Fitted parameters:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,datablock,category,entry,parameter,start,fitted,uncertainty,units,change
1,si,cell,,length_a,5.4325,5.4311,N/A,Å,0.03 % ↓
2,si,atom_site,Si,adp_iso,0.5240,0.5444,N/A,Å²,3.89 % ↑
3,sepd,linked_phases,si,scale,14.9693,1141.3765,N/A,,7524.80 % ↑
4,sepd,peak,,rise_alpha_1,0.5971,0.4806,N/A,μs/Å,19.52 % ↓
5,sepd,peak,,decay_beta_0,0.0422,0.0411,N/A,μs,2.69 % ↓
6,sepd,peak,,decay_beta_1,0.0095,0.0110,N/A,μs/Å,16.76 % ↑
7,sepd,peak,,lorentz_gamma_1,2.5489,2.0456,N/A,μs/Å,19.74 % ↓
8,sepd,peak,,gauss_sigma_0,3.0148,-0.0016,N/A,μs²,100.05 % ↓
9,sepd,peak,,gauss_sigma_1,33.3451,37.7826,N/A,μs/Å,13.31 % ↑
10,sepd,instrument,,d_to_tof_offset,-8.7248,-8.3301,N/A,μs,4.52 % ↓


#### Show parameter correlations

In [47]:
project.display.plotter.plot_param_correlations()

⚠️ Correlation matrix is unavailable for this fit. Use the lmfit minimizer and ensure covariance estimation succeeds.             


#### Plot Measured vs Calculated

In [48]:
project.display.plotter.plot_meas_vs_calc(
    expt_name='sepd', x_min=23200, x_max=23700, show_residual=True
)

In [49]:
project.display.plotter.plot_meas_vs_calc(expt_name='sepd', x='d_spacing', show_residual=True)